In [ ]:
from ROOT import TFile, TCanvas, kBlack, TPad, TGaxis, kRed, kBlue, kGray, gROOT
import uuid 
%jsroot on

fileName_rel = 'electrons632.root'
fileName_ref = 'electrons.root'

histoPath="electronValidation"

def getHisto(file, path):
    t_path = file.Get(path)
    return t_path 

fileRoot_rel = TFile(fileName_rel)
fileRoot_ref = TFile(fileName_ref)
histo_relPath = getHisto(fileRoot_rel,histoPath)
histo_refPath = getHisto(fileRoot_ref,histoPath)

In [ ]:
def check_histogram_type(hist,type):
    return hist.InheritsFrom(type)

def createHistoPicture(histo1,histo2,rescale=True,rebin=1):
    if (rebin>1):
        histo1.Rebin(rebin)
        histo2.Rebin(rebin)

    histo2.SetLineColor(2)
    normalOrder = True
    if (rescale):
        normalOrder = (histo1.GetBinContent(histo1.GetMaximumBin()))/histo1.Integral() >  (histo2.GetBinContent(histo2.GetMaximumBin()))/histo2.Integral()
    else:
        normalOrder = (histo1.GetBinContent(histo1.GetMaximumBin())) >  (histo2.GetBinContent(histo2.GetMaximumBin()))
    
    
    if (normalOrder): 
        if (rescale):
            histo1.DrawNormalized()
            histo2.DrawNormalized("same")
        else:
            histo1.Draw()
            histo2.Draw("same")
    else:
        if (rescale):
            histo2.DrawNormalized()
            histo1.DrawNormalized("same")
        else:
            histo2.Draw()
            histo1.Draw("same")
    return histo1,histo2
   
def compareHisto(histoName, rescale=True, rebin=1):
    histo_rel = histo_relPath.Get(histoName)
    histo_ref = histo_refPath.Get(histoName)
    cnv = createHistoPicture(histo_ref,histo_rel,False,rebin)
    fileName=histoName+".png"
    cnv.Print(fileName)
    return cnv

def createRatio(h1, h2):
    print("Is TProfile "+ str(check_histogram_type(h1,"TProfile")))
    h3 = h1.Clone("h3")
    h3.SetLineColor(kBlack)
    h3.SetMarkerStyle(21)
    h3.SetTitle("")
    h3.SetMinimum(0.8)
    h3.SetMaximum(1.35)
    # Set up plot for markers and errors
    h3.Sumw2()
    h3.SetStats(0)
    h3.Divide(h2)
 
    # Adjust y-axis settings
    y = h3.GetYaxis()
    y.SetTitle("Ratio ")
    y.SetNdivisions(505)
    y.SetTitleSize(20)
    y.SetTitleFont(43)
    y.SetTitleOffset(1.55)
    y.SetLabelFont(43)
    y.SetLabelSize(15)
 
    # Adjust x-axis settings
    x = h3.GetXaxis()
    x.SetTitleSize(20)
    x.SetTitleFont(43)
    x.SetTitleOffset(4.0)
    x.SetLabelFont(43)
    x.SetLabelSize(15)
 
    return h3

def createCanvasPads():
    # Check for an existing canvas and delete it if necessary
    #if gROOT.FindObject("c"):
    #   print("Explicitely Deleting canvas ")
    #    gROOT.FindObject("c").Close()
    #    del c
    canvas_name = f"c1_{uuid.uuid4().hex}" 
    c = TCanvas(canvas_name,"c" ,800, 800)
    # Upper histogram plot is pad1
    pad1 = TPad("pad1", "pad1", 0, 0.3, 1, 1.0)
    pad1.SetBottomMargin(0)  # joins upper and lower plot
    pad1.SetGridx()
    pad1.Draw()
    # Lower ratio plot is pad2
    c.cd()  # returns to main canvas before defining pad2
    pad2 = TPad("pad2", "pad2", 0, 0.05, 1, 0.3)
    pad2.SetTopMargin(0)  # joins upper and lower plot
    pad2.SetBottomMargin(0.2)
    pad2.SetGridx()
    pad2.Draw()
 
    return c, pad1, pad2

def drawRatio(histoName):
    cnvR = TCanvas("canvasratio")
    histo_rel = histo_relPath.Get(histoName)
    histo_ref = histo_refPath.Get(histoName)
    h3=createRatio(histo_rel ,histo_ref)
    h3.Draw("ep")
    
    fileName=histoName+"Ratio.png"
    cnvR.Print(fileName)
    return cnvR,h3

 
def ratioplot(histoName,rescale=True, rebin=1):
    # create required parts
    h1 = histo_relPath.Get(histoName)
    h2 = histo_refPath.Get(histoName)
    # h1 = h1r.ProjectionX()
    # h2 = h2r.ProjectionX()
    h1,h2 = createHistoPicture(h1,h2,rescale,rebin)
    h3 = createRatio(h1, h2)
    c, pad1, pad2 = createCanvasPads()
    
    # draw everything
    pad1.cd()
    h1.SetLineColor(kRed)
    h1.SetStats(1)
    h1.SetLineStyle(0)
    h1.SetLineWidth(2)
    h1.Draw()
    pad1.Update()
    statBox1 = h1.GetListOfFunctions().FindObject("stats")
    statBox1.SetTextColor(kRed)    
    statBox1.SetBorderSize(2)
    statBox1.SetFillColor(kGray)
#    statBox1.SetFillColorAlpha(18, 0.35) # https://root.cern.ch/doc/master/classTAttFill.html
    statBox1.SetY2NDC(0.995)
    statBox1.SetY1NDC(0.755)
    statBox1.SetX2NDC(0.995)
    statBox1.SetX1NDC(0.795)
    h2.SetLineColor(kBlue)
    h2.Draw("same")


    
    # to avoid clipping the bottom zero, redraw a small axis
    h1.GetYaxis().SetLabelSize(0.0)
    axis = TGaxis(-5, 20, -5, 220, 20, 220, 510, "")
    axis.SetLabelFont(43)
    axis.SetLabelSize(15)
    axis.Draw()
    pad2.cd()
    h3.Draw("ep")
    print(c)
    return c ,h3
 
#if __name__ == "__main__":
#    c,h=ratioplot("EoPvsEtaProfMax",False)
#    c.Update()    
#    c.Draw()

#histoName='dRPhoPFcand_Barrel_EtaR'
#histoName='dRPhoPFcand_Barrel_Edge'
#histoName='dRPhoPFcand_all'
#histoName='EtaPhotonsBarrel'
#histoName='EtaPhotonsCheckEdge'
#histoName='EoPvsEtaProfMax'
#canvas=compareHisto(histoName,'False')
#canvas.Draw()


#histosToPrint=[['dRPhoPFcand_Barrel_EtaR',False],['dRPhoPFcand_Barrel_Edge',True],['NeutralHadronEta',False]]

#for i in histosToPrint:
#    compareHisto(i[0],i[1])



In [33]:
from ROOT import gROOT
gROOT.Reset()
c1,h=ratioplot("EoPvsEtaProfMax",False)


Is TProfile True
Name: c1_808ff89a3b5b4c9a813de3350f308aa0 Title: c


Warning in <TProfile::Divide>: Cannot preserve during the division of profiles the sum of bin weight square


In [ ]:
c,h=ratioplot("EoPvsEtaProf",False)
c.Draw()
c.Update()

Is TProfile True
Name: c1_2e3028b7b11149e9a42d5cb0c541d1df Title: c


Warning in <TProfile::Divide>: Cannot preserve during the division of profiles the sum of bin weight square
